# Lab 4 — From one neuron to a small MLP

This lab is a guided path from **one neuron** to a small neural network.
Work from top to bottom and use **Shift + Enter** to run each cell.

**Run it:** upload this notebook to Google Colab, or follow the short
[setup guide](../../week02/SETUP.md). It uses only **NumPy**,
**Matplotlib**, and **scikit-learn**.

Complete each TODO cell, then run the check immediately below it.

By the end, you will be able to:

- compute a neuron's weighted sum and probability;
- build a dense hidden layer with ReLU;
- trace a complete forward pass;
- calculate one backward step and update every parameter;
- compare a linear classifier with a small MLP on `make_moons`.

The optional final section connects the same ideas back to regression.
Matplotlib is used only for pictures; plotting helpers are clearly marked.

## Goal and learning loop

We will repeat one loop throughout the notebook:

> input → forward pass → prediction → loss → backward pass → update

A **parameter** is a number the model can change while learning. In this
lab, the parameters are weights and biases.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

np.set_printoptions(precision=3, suppress=True)
plt.style.use("default")

BLUE = "#2563EB"
ORANGE = "#F97316"
DARK = "#1F2937"
LIGHT = "#E5E7EB"

print("NumPy version:", np.__version__)
print("Ready for Lab 4")

In [ ]:
# Visualization helper: you do not need to understand this plotting code yet.
def plot_probability_curve(sigmoid_function):
    scores = np.linspace(-6, 6, 200)
    probabilities = sigmoid_function(scores)
    fig, ax = plt.subplots(figsize=(6.5, 3.5))
    ax.plot(scores, probabilities, color=BLUE, linewidth=3)
    ax.axhline(0.5, color=DARK, linestyle="--", linewidth=1.2)
    ax.axvline(0, color=DARK, linestyle="--", linewidth=1.2)
    ax.set(
        title="Sigmoid turns scores into probabilities",
        xlabel="score",
        ylabel="probability",
        ylim=(-0.05, 1.05),
    )
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()


def plot_decision_boundary(ax, model, features, targets, title):
    x_min, x_max = features[:, 0].min() - 0.4, features[:, 0].max() + 0.4
    y_min, y_max = features[:, 1].min() - 0.4, features[:, 1].max() + 0.4
    grid_x, grid_y = np.meshgrid(
        np.linspace(x_min, x_max, 220),
        np.linspace(y_min, y_max, 220),
    )
    grid = np.column_stack([grid_x.ravel(), grid_y.ravel()])
    probabilities = model.predict_proba(grid)[:, 1].reshape(grid_x.shape)
    ax.contourf(
        grid_x,
        grid_y,
        probabilities,
        levels=[0, 0.5, 1],
        colors=["#DBEAFE", "#FFEDD5"],
        alpha=0.75,
    )
    ax.contour(
        grid_x,
        grid_y,
        probabilities,
        levels=[0.5],
        colors=[DARK],
        linewidths=2,
    )
    ax.scatter(
        features[targets == 0, 0],
        features[targets == 0, 1],
        color=BLUE,
        marker="o",
        edgecolor="white",
        label="class 0",
    )
    ax.scatter(
        features[targets == 1, 0],
        features[targets == 1, 1],
        color=ORANGE,
        marker="^",
        edgecolor="white",
        label="class 1",
    )
    ax.set(title=title, xlabel="feature 1", ylabel="feature 2")
    ax.legend(frameon=False, loc="upper right")

## Part 1 — One neuron

A neuron first calculates a weighted sum:

$$z=x_1w_1+x_2w_2+b = x\cdot w+b.$$

For binary classification, sigmoid turns the score into a probability:

$$\sigma(z)=\frac{1}{1+e^{-z}}.$$

**Visual warm-up:** use [Google's node interactive](https://developers.google.com/machine-learning/crash-course/neural-networks/nodes-hidden-layers).
Change one input, one weight, and the bias. Predict before each change.

**Worked example A.** For `x = [2, 1]`, `w = [0.5, -1]`, and
`b = 0.2`, what is the score?

<details><summary>Click for answer</summary>

`2(0.5) + 1(-1) + 0.2 = 0.2`. The weights control the contribution
of each input; the bias shifts the final score.

</details>

**Worked example B.** What are `sigmoid(0)` and its predicted class if
the rule is “class 1 when probability is at least 0.5”?

<details><summary>Click for answer</summary>

`sigmoid(0) = 0.5`, so the rule predicts class 1.

</details>

In [ ]:
# Run the two worked examples after predicting their outputs.
worked_x = np.array([2.0, 1.0])
worked_w = np.array([0.5, -1.0])
worked_b = 0.2

worked_score = worked_x @ worked_w + worked_b
worked_probability = 1 / (1 + np.exp(-worked_score))

print("score:", worked_score)
print("probability:", worked_probability)
print("class:", int(worked_probability >= 0.5))

### Task 1 — implement one neuron

Complete `linear_output` and `sigmoid`. Use NumPy operations so the
functions work for one example or many examples.

In [ ]:
# TODO 1
def linear_output(inputs, weights, bias):
    # Return inputs @ weights + bias.
    return None


def sigmoid(scores):
    # Return 1 / (1 + exp(-scores)).
    return None

In [ ]:
test_score = linear_output(
    np.array([1.0, 2.0]),
    np.array([0.5, -0.25]),
    0.2,
)
test_probabilities = sigmoid(np.array([-2.0, 0.0, 2.0]))

if test_score is None or test_probabilities is None:
    print("Complete Task 1, then run this check again.")
else:
    assert np.isclose(test_score, 0.2)
    assert np.isclose(sigmoid(0.0), 0.5)
    assert np.all((test_probabilities >= 0) & (test_probabilities <= 1))
    print("scores -> probabilities:", test_probabilities)
    print("classes:", (test_probabilities >= 0.5).astype(int))
    print("✓ One neuron works")
    plot_probability_curve(sigmoid)

**Explain it.** Which values are parameters, and what does sigmoid do?
Write one sentence before continuing.

## Part 2 — A hidden layer

A dense layer calculates many neurons at once:

$$Z=XW+b.$$

ReLU keeps positive scores and replaces negative scores with zero:

$$\operatorname{ReLU}(z)=\max(0,z).$$

**See why the bend matters:** compare a linear activation with ReLU in
[Google's neural-network exercise](https://developers.google.com/machine-learning/crash-course/neural-networks/interactive-exercises).

**Worked example A.** What is ReLU applied to `[-2, 0, 3]`?

<details><summary>Click for answer</summary>

`[0, 0, 3]`. Negative neurons are inactive; the positive neuron stays active.

</details>

**Worked example B.** `X` has shape `(4, 2)` and `W` has shape `(2, 3)`.
What shape does `X @ W` have?

<details><summary>Click for answer</summary>

`(4, 3)`: four examples, each producing three neuron scores.

</details>

In [ ]:
# Run the two worked examples after predicting their outputs.
relu_example = np.maximum(0, np.array([-2.0, 0.0, 3.0]))
shape_X = np.zeros((4, 2))
shape_W = np.zeros((2, 3))

print("ReLU example:", relu_example)
print("matrix shapes:", shape_X.shape, "@", shape_W.shape, "->", (shape_X @ shape_W).shape)

### Task 2 — implement ReLU and a dense layer

Complete both functions. The bias should be added to every example row.

In [ ]:
# TODO 2
def relu(scores):
    # Replace negative values with zero.
    return None


def dense_layer(inputs, weights, bias):
    # Calculate every neuron score, then apply ReLU.
    return None

In [ ]:
layer_inputs = np.array([
    [1.0, 2.0],
    [-1.0, 0.5],
])
layer_weights = np.array([
    [0.5, -1.0, 0.2],
    [1.0, 0.5, -0.4],
])
layer_bias = np.array([0.0, 0.2, 0.1])

relu_test = relu(np.array([-3.0, 0.0, 2.0]))
layer_output = dense_layer(layer_inputs, layer_weights, layer_bias)

if relu_test is None or layer_output is None:
    print("Complete Task 2, then run this check again.")
else:
    assert np.all(relu_test >= 0)
    assert np.allclose(relu_test, [0, 0, 2])
    assert layer_output.shape == (2, 3)
    print("X shape:", layer_inputs.shape)
    print("W shape:", layer_weights.shape)
    print("hidden output shape:", layer_output.shape)
    print("hidden outputs:\n", layer_output)
    print("✓ ReLU and the dense layer work")

**Explain it.** Why do we put ReLU between two linear layers?
Write one sentence before continuing.

## Part 3 — Forward propagation

Our small MLP has:

- 2 input features;
- 3 hidden neurons with ReLU;
- 1 output neuron with sigmoid.

**Worked example A.** If five examples enter a three-neuron hidden layer,
what is the hidden-output shape?

<details><summary>Click for answer</summary>

`(5, 3)`: one row per example and one column per hidden neuron.

</details>

**Worked example B.** A probability is `0.73`. What class is predicted at
a threshold of `0.5`?

<details><summary>Click for answer</summary>

Class 1, because `0.73 >= 0.5`.

</details>

In [ ]:
# Fixed parameters make this forward-pass exercise reproducible.
forward_inputs = np.array([
    [0.2, 0.8],
    [1.0, -0.5],
    [-0.7, 0.3],
    [0.6, 1.2],
    [-0.4, -0.9],
])
forward_W1 = np.array([
    [0.7, -0.2, 0.4],
    [0.1, 0.8, -0.5],
])
forward_b1 = np.array([0.0, 0.1, 0.2])
forward_W2 = np.array([
    [0.9],
    [-0.6],
    [0.7],
])
forward_b2 = np.array([0.05])

print("inputs:", forward_inputs.shape)
print("W1:", forward_W1.shape, "W2:", forward_W2.shape)

### Task 3 — complete the forward pass

Return both the probabilities and a `cache`. The cache remembers values
that the backward pass will need later.

In [ ]:
# TODO 3
def mlp_forward(inputs, W1, b1, W2, b2):
    hidden_scores = None
    hidden_outputs = None
    output_scores = None
    probabilities = None

    cache = (inputs, hidden_scores, hidden_outputs, probabilities)
    return probabilities, cache

In [ ]:
forward_probabilities, forward_cache = mlp_forward(
    forward_inputs,
    forward_W1,
    forward_b1,
    forward_W2,
    forward_b2,
)

if forward_probabilities is None:
    print("Complete Task 3, then run this check again.")
else:
    _, hidden_scores, hidden_outputs, _ = forward_cache
    assert hidden_scores.shape == (5, 3)
    assert hidden_outputs.shape == (5, 3)
    assert forward_probabilities.shape == (5, 1)
    assert np.all((forward_probabilities >= 0) & (forward_probabilities <= 1))

    print("first input:", forward_inputs[0])
    print("first hidden scores:", hidden_scores[0])
    print("first hidden activations:", hidden_outputs[0])
    print("first probability:", forward_probabilities[0, 0])
    print("all predicted classes:", (forward_probabilities[:, 0] >= 0.5).astype(int))
    print("✓ The complete forward pass works")

**Explain it.** What is the difference between a hidden score and a
hidden activation? Write one sentence.

## Part 4 — One backward step

Loss measures how wrong the predictions are. For binary classification,
we use binary cross-entropy. You do **not** need to derive it here.

Backpropagation asks: **How much did each parameter contribute to the
error?** The gradient of a parameter must have the same shape as that
parameter.

**Optional visual trace:** [Backprop Explainer](https://xnought.github.io/backprop-explainer/)
shows the forward and backward directions. Focus on direction, not every formula.

**Worked example A.** If probability is `0.8` and the target is `1`, the
output error is `probability - target`. What is it?

<details><summary>Click for answer</summary>

`0.8 - 1 = -0.2`. The negative sign says increasing the score would reduce
the error for this example.

</details>

**Worked example B.** Hidden scores are `[-1, 0.4, 2]`. What is the ReLU
mask `hidden_scores > 0`?

<details><summary>Click for answer</summary>

`[False, True, True]`. Only the two positive neurons pass a backward signal.

</details>

In [ ]:
def binary_cross_entropy(probabilities, targets):
    probabilities = np.clip(probabilities.reshape(-1), 1e-9, 1 - 1e-9)
    targets = targets.reshape(-1)
    return -np.mean(
        targets * np.log(probabilities)
        + (1 - targets) * np.log(1 - probabilities)
    )


# A tiny fixed batch and fixed parameters for one update.
backward_inputs = np.array([
    [-1.0, -0.5],
    [-0.8, 0.2],
    [-0.2, -0.9],
    [0.5, 0.7],
    [0.8, -0.1],
    [1.1, 0.9],
])
backward_targets = np.array([0, 0, 0, 1, 1, 1])

initial_W1 = np.array([
    [0.35, -0.45, 0.25],
    [0.20, 0.30, -0.40],
])
initial_b1 = np.array([0.10, 0.05, 0.15])
initial_W2 = np.array([[0.50], [-0.35], [0.40]])
initial_b2 = np.array([0.0])

initial_probabilities, initial_cache = mlp_forward(
    backward_inputs,
    initial_W1,
    initial_b1,
    initial_W2,
    initial_b2,
)

if initial_probabilities is None:
    initial_loss = None
    print("Complete Task 3 first, then return here.")
else:
    initial_loss = binary_cross_entropy(initial_probabilities, backward_targets)
    print("loss before the update:", initial_loss)

### Task 4 — calculate every gradient

Complete the four missing gradient calculations. Read each line by its
shapes, not as a formula to memorize.

In [ ]:
# TODO 4
def mlp_gradients(targets, W2, cache):
    inputs, hidden_scores, hidden_outputs, probabilities = cache
    number_of_examples = len(inputs)

    output_error = (probabilities - targets.reshape(-1, 1)) / number_of_examples

    gradient_W2 = None
    gradient_b2 = None

    hidden_error = output_error @ W2.T
    relu_mask = hidden_scores > 0
    hidden_score_error = hidden_error * relu_mask

    gradient_W1 = None
    gradient_b1 = None

    return gradient_W1, gradient_b1, gradient_W2, gradient_b2

In [ ]:
if initial_loss is None:
    print("Complete the forward pass before checking Task 4.")
else:
    gradients = mlp_gradients(backward_targets, initial_W2, initial_cache)

    if any(gradient is None for gradient in gradients):
        print("Complete Task 4, then run this check again.")
    else:
        gradient_W1, gradient_b1, gradient_W2, gradient_b2 = gradients

        assert gradient_W1.shape == initial_W1.shape
        assert gradient_b1.shape == initial_b1.shape
        assert gradient_W2.shape == initial_W2.shape
        assert gradient_b2.shape == initial_b2.shape

        learning_rate = 0.8
        updated_W1 = initial_W1 - learning_rate * gradient_W1
        updated_b1 = initial_b1 - learning_rate * gradient_b1
        updated_W2 = initial_W2 - learning_rate * gradient_W2
        updated_b2 = initial_b2 - learning_rate * gradient_b2

        updated_probabilities, _ = mlp_forward(
            backward_inputs,
            updated_W1,
            updated_b1,
            updated_W2,
            updated_b2,
        )
        updated_loss = binary_cross_entropy(updated_probabilities, backward_targets)

        print("W1 and gradient_W1:", initial_W1.shape, gradient_W1.shape)
        print("b1 and gradient_b1:", initial_b1.shape, gradient_b1.shape)
        print("W2 and gradient_W2:", initial_W2.shape, gradient_W2.shape)
        print("b2 and gradient_b2:", initial_b2.shape, gradient_b2.shape)
        print("loss before:", initial_loss)
        print("loss after: ", updated_loss)
        assert updated_loss < initial_loss
        print("✓ Every gradient matches its parameter, and one update lowers loss")

**Explain it.** In plain language, what moved backward? Why did we
subtract the gradients from the parameters?

## Part 5 — Train full models with scikit-learn

Now scikit-learn will repeat the forward pass, loss calculation,
backpropagation, and parameter updates for us.

We compare:

- logistic regression: a linear classification boundary;
- an MLP with one hidden layer: a nonlinear boundary.

**Interactive challenge:** use [TensorFlow Playground](https://playground.tensorflow.org/)
to compare Linear and ReLU on XOR or spiral data. Hover over each hidden
neuron and describe the region it creates.

**Worked example A.** What is the difference between `fit` and `predict`?

<details><summary>Click for answer</summary>

`fit(X, y)` learns parameters from labeled examples. `predict(X)` uses
already learned parameters to assign classes.

</details>

**Worked example B.** Why put `StandardScaler()` inside a pipeline?

<details><summary>Click for answer</summary>

It scales features before the model sees them and applies the same learned
scaling when we later call `predict`.

</details>

In [ ]:
moon_features, moon_targets = make_moons(
    n_samples=240,
    noise=0.18,
    random_state=7,
)
X_train, X_test, y_train, y_test = train_test_split(
    moon_features,
    moon_targets,
    test_size=0.30,
    random_state=11,
    stratify=moon_targets,
)

print("training inputs:", X_train.shape)
print("test inputs:", X_test.shape)
print("training targets:", y_train.shape)

### Task 5 — fit a linear model and a small MLP

Build both pipelines, then call `fit` before `predict`. Keep the supplied
settings so your result matches the expected accuracy ranges.

In [ ]:
# TODO 5
logistic_model = None
mlp_model = None

# 1. Create a pipeline with StandardScaler and LogisticRegression.
# 2. Create a pipeline with StandardScaler and MLPClassifier.
# 3. Fit both models using X_train and y_train.

In [ ]:
if logistic_model is None or mlp_model is None:
    print("Complete Task 5, then run this check again.")
else:
    try:
        logistic_predictions = logistic_model.predict(X_test)
        mlp_predictions = mlp_model.predict(X_test)
    except Exception as error:
        print("Call fit on both models before predict.")
        print("Model message:", error)
    else:
        logistic_accuracy = np.mean(logistic_predictions == y_test)
        mlp_accuracy = np.mean(mlp_predictions == y_test)

        assert 0.75 <= logistic_accuracy <= 0.95
        assert 0.88 <= mlp_accuracy <= 1.00
        assert mlp_accuracy > logistic_accuracy

        improved_indices = np.flatnonzero(
            (logistic_predictions != y_test) & (mlp_predictions == y_test)
        )
        assert len(improved_indices) > 0
        improved_index = improved_indices[0]

        print(f"logistic test accuracy: {logistic_accuracy:.3f}")
        print(f"MLP test accuracy:      {mlp_accuracy:.3f}")
        print("example improved test point:", X_test[improved_index])
        print("true class:", y_test[improved_index])
        print("logistic prediction:", logistic_predictions[improved_index])
        print("MLP prediction:", mlp_predictions[improved_index])
        print("✓ The nonlinear MLP improves on the linear boundary")

In [ ]:
if logistic_model is None or mlp_model is None:
    print("Fit both models in Task 5 to reveal the decision boundaries.")
else:
    try:
        logistic_model.predict(X_test[:1])
        mlp_model.predict(X_test[:1])
    except Exception:
        print("Fit both models in Task 5 to reveal the decision boundaries.")
    else:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True, sharey=True)
        plot_decision_boundary(
            axes[0],
            logistic_model,
            moon_features,
            moon_targets,
            "Logistic regression: linear boundary",
        )
        plot_decision_boundary(
            axes[1],
            mlp_model,
            moon_features,
            moon_targets,
            "Small MLP: nonlinear boundary",
        )
        plt.tight_layout()
        plt.show()

**Explain it.** Describe one visible difference between the two
boundaries. Why can the MLP make that shape?

## Part 6 — Optional regression connection

Classification and regression use the same learning loop. The main change
is the output: regression predicts a number instead of a class.

**Worked example A.** A straight-line model sees curved data. What kind of
mistake should you expect?

<details><summary>Click for answer</summary>

It will miss the bends in a systematic way because one line cannot follow
the curve.

</details>

**Worked example B.** Does `MLPRegressor.fit(X, y)` still learn weights and
biases?

<details><summary>Click for answer</summary>

Yes. The model still learns parameters through repeated loss, gradient,
and update steps; it simply predicts continuous numbers.

</details>

In [ ]:
regression_rng = np.random.default_rng(3)
regression_inputs = np.linspace(-2.5, 2.5, 70).reshape(-1, 1)
regression_targets = (
    0.25 * regression_inputs[:, 0]
    + 0.8 * np.sin(2 * regression_inputs[:, 0])
    + regression_rng.normal(0, 0.08, len(regression_inputs))
)

print("regression inputs:", regression_inputs.shape)
print("regression targets:", regression_targets.shape)

### Optional Task 6 — fit a line and a regression MLP

Create and fit `LinearRegression` and the supplied `MLPRegressor` pipeline.

In [ ]:
# TODO 6 (optional)
line_model = None
curve_model = None

# Create both models, then fit them to regression_inputs and
# regression_targets.

In [ ]:
if line_model is None or curve_model is None:
    print("Optional: complete Task 6 to compare regression models.")
else:
    try:
        line_predictions = line_model.predict(regression_inputs)
        curve_predictions = curve_model.predict(regression_inputs)
    except Exception as error:
        print("Call fit on both regression models before predict.")
        print("Model message:", error)
    else:
        line_mse = np.mean((line_predictions - regression_targets) ** 2)
        curve_mse = np.mean((curve_predictions - regression_targets) ** 2)
        assert curve_mse < line_mse

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.scatter(
            regression_inputs[:, 0],
            regression_targets,
            color=DARK,
            s=28,
            alpha=0.65,
            label="data",
        )
        ax.plot(
            regression_inputs[:, 0],
            line_predictions,
            color=BLUE,
            linewidth=2.5,
            linestyle="--",
            label=f"linear model (MSE {line_mse:.3f})",
        )
        ax.plot(
            regression_inputs[:, 0],
            curve_predictions,
            color=ORANGE,
            linewidth=3,
            label=f"MLP (MSE {curve_mse:.3f})",
        )
        ax.set(
            title="Linear and MLP regression predictions",
            xlabel="input",
            ylabel="target",
        )
        ax.legend(frameon=False)
        ax.grid(alpha=0.2)
        plt.tight_layout()
        plt.show()
        print("✓ The hidden layers follow the curved pattern")

**Explain it.** What changed between classification and regression,
and what stayed the same?

## Final check

Without looking back, answer these questions:

1. What does one neuron calculate before its activation?
2. Why does an MLP need a nonlinear activation such as ReLU?
3. What values are stored during the forward pass for backpropagation?
4. Why must each gradient have the same shape as its parameter?
5. What is the difference between `fit` and `predict`?

Then write the complete learning loop one final time.

## Next steps

Try one small change at a time:

- change the classification MLP from 8 hidden neurons to 4 or 16;
- increase the moon noise and observe both boundaries;
- change the learning rate in the manual update;
- print one hidden activation and explain when that neuron is active.

The next lecture will extend this same forward-and-backward story through
deeper networks.